In [6]:
import pandas as pd
import numpy as np


# ============================================================
# 1. LOAD DATA
# ============================================================

file_path = "/Users/velagapudiruthvika/Downloads/Lab Session Data (1) (3).xlsx"

df = pd.read_excel(file_path, sheet_name="thyroid0387_UCI")

# Replace '?' with missing values
df = df.replace("?", np.nan)

# Target column
target_column = "Condition"

# Remove Record ID because it is only an identifier
df = df.drop(columns=["Record ID"])


# ============================================================
# 2. DATA IMPUTATION
# ============================================================

def impute_missing_values(data):
    """
    Fill missing numerical values using median
    and categorical values using mode.
    """

    data = data.copy()

    numerical_columns = data.select_dtypes(
        include=[np.number]
    ).columns

    categorical_columns = data.select_dtypes(
        exclude=[np.number]
    ).columns

    # Numerical data -> median
    for column in numerical_columns:
        data[column] = data[column].fillna(
            data[column].median()
        )

    # Categorical data -> mode
    for column in categorical_columns:
        if data[column].isna().any():
            data[column] = data[column].fillna(
                data[column].mode()[0]
            )

    return data


# ============================================================
# 3. ENCODING
# ============================================================

def encode_categorical_data(data, target_column):
    """
    Convert categorical feature values into numerical values.
    The target column is kept separately.
    """

    data = data.copy()

    feature_columns = [
        col for col in data.columns
        if col != target_column
    ]

    encoders = {}

    for column in feature_columns:

        if data[column].dtype == "object":

            categories = sorted(
                data[column].unique()
            )

            mapping = {
                category: index
                for index, category in enumerate(categories)
            }

            data[column] = data[column].map(mapping)

            encoders[column] = mapping

    return data, encoders


# ============================================================
# 4. DISTANCE CALCULATION
# ============================================================

def calculate_distance(point1, point2, metric="euclidean"):
    """
    Calculate distance between two data points.

    metric:
        euclidean -> Euclidean distance
        manhattan -> Manhattan distance
    """

    point1 = np.asarray(point1, dtype=float)
    point2 = np.asarray(point2, dtype=float)

    if metric == "euclidean":

        return np.sqrt(
            np.sum((point1 - point2) ** 2)
        )

    elif metric == "manhattan":

        return np.sum(
            np.abs(point1 - point2)
        )

    else:
        raise ValueError(
            "Choose 'euclidean' or 'manhattan'"
        )


# ============================================================
# 5. SORTING
# ============================================================

def sort_distances(distance_list, algorithm="bubble"):
    """
    Sort (distance, index) pairs.

    algorithm:
        bubble    -> Bubble Sort
        selection -> Selection Sort
    """

    arr = distance_list.copy()

    if algorithm == "bubble":

        n = len(arr)

        for i in range(n):

            for j in range(0, n - i - 1):

                if arr[j][0] > arr[j + 1][0]:

                    arr[j], arr[j + 1] = (
                        arr[j + 1],
                        arr[j]
                    )

    elif algorithm == "selection":

        n = len(arr)

        for i in range(n):

            minimum = i

            for j in range(i + 1, n):

                if arr[j][0] < arr[minimum][0]:
                    minimum = j

            arr[i], arr[minimum] = (
                arr[minimum],
                arr[i]
            )

    else:
        raise ValueError(
            "Choose 'bubble' or 'selection'"
        )

    return arr


# ============================================================
# 6. IDENTIFY K NEAREST NEIGHBORS
# ============================================================

def identify_neighbors(
    X_train,
    test_point,
    k=3,
    metric="euclidean",
    sorting_algorithm="bubble"
):
    """
    Find k nearest neighbors.

    Equal-distance ties are resolved using
    the training-data index.
    """

    distances = []

    for index, train_point in enumerate(X_train):

        distance = calculate_distance(
            train_point,
            test_point,
            metric
        )

        distances.append(
            (distance, index)
        )

    # Sorting is stable and index is used
    # as a tie breaker for equal distances.
    distances.sort(
        key=lambda x: (x[0], x[1])
    )

    # Use our own sorting algorithm
    sorted_distances = sort_distances(
        distances,
        sorting_algorithm
    )

    return sorted_distances[:k]


# ============================================================
# 7. CLASS EVALUATION / ASSIGNMENT
# ============================================================

def assign_class(neighbors, y_train):
    """
    Assign a class using majority voting.

    If there is a tie:
    1. Choose the class having the smallest
       total distance among tied classes.
    2. If still tied, choose alphabetically.
    """

    votes = {}

    distance_sum = {}

    for distance, index in neighbors:

        label = y_train[index]

        votes[label] = votes.get(label, 0) + 1

        distance_sum[label] = (
            distance_sum.get(label, 0)
            + distance
        )

    # Maximum number of votes
    maximum_votes = max(votes.values())

    tied_classes = [
        label
        for label, count in votes.items()
        if count == maximum_votes
    ]

    # No tie
    if len(tied_classes) == 1:
        return tied_classes[0]

    # Tie-breaking using total distance
    winner = min(
        tied_classes,
        key=lambda label: (
            distance_sum[label],
            str(label)
        )
    )

    return winner


# ============================================================
# 8. COMPLETE kNN PREDICTION
# ============================================================

def knn_predict(
    X_train,
    y_train,
    test_point,
    k=3,
    metric="euclidean",
    sorting_algorithm="bubble"
):
    """
    Complete kNN prediction for one test vector.
    """

    neighbors = identify_neighbors(
        X_train,
        test_point,
        k,
        metric,
        sorting_algorithm
    )

    predicted_class = assign_class(
        neighbors,
        y_train
    )

    return predicted_class, neighbors


# ============================================================
# 9. PREPROCESS DATA
# ============================================================

df = impute_missing_values(df)

df, encoders = encode_categorical_data(
    df,
    target_column
)

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column].values

# Convert to NumPy arrays
X = X.astype(float).values


# ============================================================
# 10. TEST THE MODULAR kNN
# ============================================================

# Use the first 90% as training data
# and remaining 10% as test data temporarily.
# A3 will later use train_test_split properly.

split_index = int(0.9 * len(X))

X_train = X[:split_index]
y_train = y[:split_index]

X_test = X[split_index:]
y_test = y[split_index:]


# Predict the first test vector
prediction, neighbors = knn_predict(
    X_train,
    y_train,
    X_test[0],
    k=3,
    metric="euclidean",
    sorting_algorithm="bubble"
)


# ============================================================
# 11. DISPLAY RESULT
# ============================================================

print("Number of features:", X.shape[1])
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nActual class:", y_test[0])
print("Predicted class:", prediction)

print("\nNearest Neighbors:")

for distance, index in neighbors:
    print(
        "Index:", index,
        "| Distance:", round(distance, 4),
        "| Class:", y_train[index]
    )

/var/folders/d0/0cx556_17ys3sv0pt_6s8slh0000gn/T/ipykernel_1541/3336828985.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("?", np.nan)


Number of features: 29
Training samples: 8254
Testing samples: 918

Actual class: NO CONDITION
Predicted class: F

Nearest Neighbors:
Index: 4247 | Distance: 42.9304 | Class: F
Index: 6321 | Distance: 45.0408 | Class: F
Index: 3509 | Distance: 45.7186 | Class: F


In [7]:
# ============================================================
# A2 - WEIGHTED kNN CLASSIFICATION
# ============================================================

def assign_weighted_class(neighbors, y_train):
    """
    Assign class using weighted voting.

    Closer neighbors receive higher weights.
    Weight = 1 / (distance + epsilon)

    Tie is resolved using the total weight and
    then alphabetically.
    """

    class_weights = {}

    epsilon = 1e-10

    for distance, index in neighbors:

        label = y_train[index]

        # Calculate weight
        weight = 1 / (distance + epsilon)

        # Add weight to the corresponding class
        class_weights[label] = (
            class_weights.get(label, 0) + weight
        )

    # Find maximum total weight
    maximum_weight = max(class_weights.values())

    # Find classes having maximum weight
    tied_classes = [
        label
        for label, weight in class_weights.items()
        if np.isclose(weight, maximum_weight)
    ]

    # No tie
    if len(tied_classes) == 1:
        return tied_classes[0]

    # Tie-breaking
    return sorted(
        tied_classes,
        key=lambda x: str(x)
    )[0]


# ============================================================
# WEIGHTED kNN PREDICTION
# ============================================================

def weighted_knn_predict(
    X_train,
    y_train,
    test_point,
    k=3,
    metric="euclidean",
    sorting_algorithm="bubble"
):
    """
    Predict the class using weighted kNN.
    """

    # Find k nearest neighbors using A1 module
    neighbors = identify_neighbors(
        X_train,
        test_point,
        k,
        metric,
        sorting_algorithm
    )

    # Assign class using weighted voting
    predicted_class = assign_weighted_class(
        neighbors,
        y_train
    )

    return predicted_class, neighbors


# ============================================================
# TEST WEIGHTED kNN
# ============================================================

prediction_weighted, weighted_neighbors = weighted_knn_predict(
    X_train,
    y_train,
    X_test[0],
    k=3,
    metric="euclidean",
    sorting_algorithm="bubble"
)


# ============================================================
# DISPLAY RESULT
# ============================================================

print("Actual class:", y_test[0])
print("Weighted kNN predicted class:", prediction_weighted)

print("\nNearest Neighbors:")

for distance, index in weighted_neighbors:

    weight = 1 / (distance + 1e-10)

    print(
        "Index:", index,
        "| Distance:", round(distance, 4),
        "| Weight:", round(weight, 6),
        "| Class:", y_train[index]
    )

Actual class: NO CONDITION
Weighted kNN predicted class: F

Nearest Neighbors:
Index: 4247 | Distance: 42.9304 | Weight: 0.023294 | Class: F
Index: 6321 | Distance: 45.0408 | Weight: 0.022202 | Class: F
Index: 3509 | Distance: 45.7186 | Weight: 0.021873 | Class: F


In [9]:
# ============================================================
# A3 - TRAIN TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("Total samples:", len(X))
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("\nTraining labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

Total samples: 9172
Training samples: 6420
Testing samples: 2752

Training data shape: (6420, 29)
Testing data shape: (2752, 29)

Training labels shape: (6420,)
Testing labels shape: (2752,)


In [10]:
# ============================================================
# A4 - kNN CLASSIFIER USING SKLEARN
# ============================================================

import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# Create kNN classifier with k = 3
neigh = KNeighborsClassifier(n_neighbors=3)

# Train the classifier
neigh.fit(X_train, y_train)

print("kNN classifier created successfully!")
print("Number of neighbors (k):", neigh.n_neighbors)

kNN classifier created successfully!
Number of neighbors (k): 3


In [11]:
# ============================================================
# A5 - TEST THE ACCURACY OF kNN
# ============================================================

accuracy = neigh.score(X_test, y_test)

print("Accuracy of kNN:", accuracy)
print("Accuracy percentage:", round(accuracy * 100, 2), "%")

Accuracy of kNN: 0.7776162790697675
Accuracy percentage: 77.76 %


In [12]:
# ============================================================
# A6 - PREDICT TEST VECTORS
# ============================================================

y_pred = neigh.predict(X_test)

print("Predicted classes:")
print(y_pred)

print("\nNumber of predictions:", len(y_pred))

Predicted classes:
['NO CONDITION' 'NO CONDITION' 'K' ... 'I' 'NO CONDITION' 'NO CONDITION']

Number of predictions: 2752


In [ ]:
# ============================================================
# A7 - DEVELOPED kNN PACKAGE
# ============================================================

import numpy as np


class MyKNN:

    def __init__(
        self,
        n_neighbors=3,
        metric="euclidean",
        sorting_algorithm="bubble"
    ):
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.sorting_algorithm = sorting_algorithm

        self.X_train = None
        self.y_train = None


    # --------------------------------------------------------
    # FIT
    # --------------------------------------------------------

    def Fit(self, X, y):
        """
        Store the training data.
        """

        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)

        return self


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    def Predict(self, X):
        """
        Predict classes for test vectors.
        """

        X = np.asarray(X, dtype=float)

        predictions = []

        for test_point in X:

            neighbors = identify_neighbors(
                self.X_train,
                test_point,
                self.n_neighbors,
                self.metric,
                self.sorting_algorithm
            )

            predicted_class = assign_class(
                neighbors,
                self.y_train
            )

            predictions.append(predicted_class)

        return np.array(predictions)


    # --------------------------------------------------------
    # SCORE
    # --------------------------------------------------------

    def Score(self, X, y):
        """
        Calculate classification accuracy.
        """

        predictions = self.Predict(X)

        y = np.asarray(y)

        accuracy = np.mean(predictions == y)

        return accuracy


# ============================================================
# CREATE OUR OWN kNN CLASSIFIER
# ============================================================

my_knn = MyKNN(
    n_neighbors=3,
    metric="euclidean",
    sorting_algorithm="bubble"
)


# ============================================================
# FIT MODEL
# ============================================================

my_knn.Fit(X_train, y_train)


# ============================================================
# PREDICT TEST DATA
# ============================================================

my_predictions = my_knn.Predict(X_test)


# ============================================================
# CALCULATE SCORE
# ============================================================

my_accuracy = my_knn.Score(
    X_test,
    y_test
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("Number of test samples:", len(X_test))

print("\nFirst 20 predictions:")

for i in range(20):
    print(
        "Test sample:", i,
        "| Actual:", y_test[i],
        "| Predicted:", my_predictions[i]
    )

print("\nOur kNN Accuracy:", my_accuracy)
print("Our kNN Accuracy (%):", round(my_accuracy * 100, 2), "%")

In [ ]:
# ============================================================
# A8 - COMPARISON: MY kNN vs SKLEARN kNN
# ============================================================

import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

# Different k values
k_values = [1, 3, 5, 7, 9]

my_accuracies = []
sklearn_accuracies = []


for k in k_values:

    print("Testing k =", k)

    # --------------------------------------------------------
    # MY kNN
    # --------------------------------------------------------

    my_model = MyKNN(n_neighbors=k)

    my_model.Fit(X_train, y_train)

    # Use a smaller test set because our implementation
    # is written from scratch and is slower.
    my_accuracy = my_model.Score(
        X_test[:500],
        y_test[:500]
    )

    my_accuracies.append(my_accuracy)


    # --------------------------------------------------------
    # SKLEARN kNN
    # --------------------------------------------------------

    sklearn_model = KNeighborsClassifier(
        n_neighbors=k
    )

    sklearn_model.fit(X_train, y_train)

    sklearn_accuracy = sklearn_model.score(
        X_test[:500],
        y_test[:500]
    )

    sklearn_accuracies.append(sklearn_accuracy)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\nComparison of Accuracy")
print("-" * 45)

for i in range(len(k_values)):

    print(
        "k =", k_values[i],
        "| My kNN:",
        round(my_accuracies[i] * 100, 2), "%",
        "| sklearn:",
        round(sklearn_accuracies[i] * 100, 2), "%"
    )


# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    my_accuracies,
    marker="o",
    label="My kNN"
)

plt.plot(
    k_values,
    sklearn_accuracies,
    marker="s",
    label="sklearn kNN"
)

plt.xlabel("Value of k")
plt.ylabel("Accuracy")

plt.title("Comparison of My kNN and sklearn kNN")

plt.xticks(k_values)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# A9 - WEIGHTED kNN COMPARISON
# ============================================================

weighted_accuracies = []

# Use the same k values as A8
for k in k_values:

    print("Testing weighted k =", k)

    # --------------------------------------------------------
    # Weighted kNN
    # --------------------------------------------------------

    weighted_predictions = []

    # Use the same 500 test samples as A8
    for test_point in X_test[:500]:

        neighbors = identify_neighbors(
            X_train,
            test_point,
            k=k,
            metric="euclidean",
            sorting_algorithm="bubble"
        )

        prediction = assign_weighted_class(
            neighbors,
            y_train
        )

        weighted_predictions.append(prediction)

    weighted_predictions = np.array(weighted_predictions)

    accuracy = np.mean(
        weighted_predictions == y_test[:500]
    )

    weighted_accuracies.append(accuracy)


# ============================================================
# DISPLAY A9 RESULTS
# ============================================================

print("\nA9 - Weighted kNN Results")
print("-" * 45)

for i in range(len(k_values)):

    print(
        "k =", k_values[i],
        "| Weighted kNN:",
        round(weighted_accuracies[i] * 100, 2), "%"
    )


# ============================================================
# COMPARISON WITH A8
# ============================================================

print("\nComplete Comparison")
print("-" * 75)

print(
    "k\tMy kNN\t\tSklearn kNN\tWeighted kNN"
)

for i in range(len(k_values)):

    print(
        k_values[i],
        "\t",
        round(my_accuracies[i] * 100, 2), "%",
        "\t\t",
        round(sklearn_accuracies[i] * 100, 2), "%",
        "\t\t",
        round(weighted_accuracies[i] * 100, 2), "%"
    )


# ============================================================
# PLOT
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    k_values,
    my_accuracies,
    marker="o",
    label="My kNN"
)

plt.plot(
    k_values,
    sklearn_accuracies,
    marker="s",
    label="sklearn kNN"
)

plt.plot(
    k_values,
    weighted_accuracies,
    marker="^",
    label="Weighted kNN"
)

plt.xlabel("Value of k")
plt.ylabel("Accuracy")

plt.title(
    "Comparison of Normal, sklearn and Weighted kNN"
)

plt.xticks(k_values)

plt.legend()

plt.grid(True)

plt.show()